## Notebook 07: Model Evaluation

### Objective
This notebook evaluates the final saved model on test data, generates detailed performance metrics, and creates deployment-ready documentation.

### Steps Covered
1. Load the saved model and preprocessor
2. Evaluate on test data
3. Generate classification report
4. Create confusion matrix
5. Calculate business metrics
6. Save evaluation report

### Importing libraries and loading the saved model

In [8]:
### Importing libraries and loading the saved model

import numpy as np
import pandas as pd
import joblib
import os
import json
from sklearn.metrics import recall_score, precision_score, accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("-"*140)
print("LOADING SAVED MODEL AND DATA")
print("-"*140)

# Load the model
model = joblib.load("../outputs/models/best_model.pkl")
print("✓ Model loaded successfully")

# Load the test data
X_test = np.load("../outputs/cleaned_data/X_test.npy")
y_test = np.load("../outputs/cleaned_data/y_test.npy")

# Load preprocessor if exists
try:
    preprocessor = joblib.load("../outputs/models/preprocessor.pkl")
    print("✓ Preprocessor loaded successfully")
except:
    print("⚠ Preprocessor not found")
    preprocessor = None

print(f"\nTest data shape: {X_test.shape}")
print(f"Test target shape: {y_test.shape}")
print(f"Test - Yes percentage: {y_test.mean() * 100:.2f}%")

--------------------------------------------------------------------------------------------------------------------------------------------
LOADING SAVED MODEL AND DATA
--------------------------------------------------------------------------------------------------------------------------------------------
✓ Model loaded successfully
✓ Preprocessor loaded successfully

Test data shape: (9043, 50)
Test target shape: (9043,)
Test - Yes percentage: 11.70%


### Making predictions on test data

In [9]:
### Making predictions on test data

print("-"*140)
print("MAKING PREDICTIONS")
print("-"*140)

# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability of YES

print(f"Predictions completed")
print(f"  Predicted YES: {(y_pred == 1).sum()}")
print(f"  Predicted NO:  {(y_pred == 0).sum()}")
print(f"  Actual YES:    {(y_test == 1).sum()}")
print(f"  Actual NO:     {(y_test == 0).sum()}")

--------------------------------------------------------------------------------------------------------------------------------------------
MAKING PREDICTIONS
--------------------------------------------------------------------------------------------------------------------------------------------
Predictions completed
  Predicted YES: 4177
  Predicted NO:  4866
  Actual YES:    1058
  Actual NO:     7985


### Calculating performance metrics

In [10]:
### Calculating performance metrics

print("-"*140)
print("PERFORMANCE METRICS")
print("-"*140)

# Calculate metrics
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Test Recall:    {recall:.4f} ({(recall*100):.1f}%)")
print(f"Test Precision: {precision:.4f} ({(precision*100):.1f}%)")
print(f"Test Accuracy:  {accuracy:.4f} ({(accuracy*100):.1f}%)")
print(f"Test F1-Score:  {f1:.4f}")

print("\n" + "-"*140)
print("CLASSIFICATION REPORT")
print("-"*140)
print(classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)']))

--------------------------------------------------------------------------------------------------------------------------------------------
PERFORMANCE METRICS
--------------------------------------------------------------------------------------------------------------------------------------------
Test Recall:    0.9650 (96.5%)
Test Precision: 0.2444 (24.4%)
Test Accuracy:  0.6469 (64.7%)
Test F1-Score:  0.3901

--------------------------------------------------------------------------------------------------------------------------------------------
CLASSIFICATION REPORT
--------------------------------------------------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

      No (0)       0.99      0.60      0.75      7985
     Yes (1)       0.24      0.97      0.39      1058

    accuracy                           0.65      9043
   macro avg       0.62      0.78      0.57      9043
weigh

### Creating confusion matrix

In [11]:
### Creating confusion matrix

print("-"*140)
print("CONFUSION MATRIX")
print("-"*140)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print("                 Predicted NO    Predicted YES")
print(f"Actual NO        {cm[0,0]:>12}    {cm[0,1]:>12}")
print(f"Actual YES       {cm[1,0]:>12}    {cm[1,1]:>12}")

print("\n" + "="*70)
print("HOW TO INTERPRET THE CONFUSION MATRIX")
print("="*70)
print("""
The confusion matrix shows four types of predictions:

+-----------------------------------------------------------+
|                       PREDICTED                           |
|                     NO           YES                      |
+-----------------------------------------------------------+
| ACTUAL    NO    [A]           [B]                         |
|           YES   [C]           [D]                         |
+-----------------------------------------------------------+

WHERE:
- [A] True Negatives: Correctly predicted NO (did not subscribe)
- [B] False Positives: Wrongly predicted YES (wasted calls)
- [C] False Negatives: Missed YES (lost opportunities)
- [D] True Positives: Correctly predicted YES (successful calls)

BUSINESS MEANING OF EACH CELL:
""")

print(f"""
+-----------------------------------------------------------+
|                    OUR MODEL'S RESULTS                    |
+-----------------------------------------------------------+
|                                 Predicted                 |
|                     NO                    YES             |
+-----------------------------------------------------------+
| ACTUAL    NO        {cm[0,0]:>10}              {cm[0,1]:>10}             |
|           YES       {cm[1,0]:>10}              {cm[1,1]:>10}             |
+-----------------------------------------------------------+

INTERPRETATION:
""")

print(f"""
1. True Negatives ({cm[0,0]}):
   -> These customers correctly predicted as NO
   -> Bank saves time by NOT calling them
   -> This is GOOD

2. False Positives ({cm[0,1]}):
   -> These customers were predicted YES but actually said NO
   -> Bank WASTED calls on these people
   -> This is the COST of the model

3. False Negatives ({cm[1,0]}):
   -> These customers were predicted NO but actually would say YES
   -> Bank MISSED these opportunities
   -> This is the RISK of the model

4. True Positives ({cm[1,1]}):
   -> These customers correctly predicted as YES
   -> Bank SUCCESSFULLY found these subscribers
   -> This is the BENEFIT of the model
""")

print("="*70)
print("BUSINESS SUMMARY")
print("="*70)

total_calls = cm[0][1] + cm[1][1]
found_yes = cm[1][1]
wasted_calls = cm[0][1]
missed_yes = cm[1][0]

print(f"""
If the bank makes {total_calls} calls based on model predictions:

- Found {found_yes} customers who actually subscribed
- Wasted {wasted_calls} calls on uninterested people
- Missed {missed_yes} interested customers (will not be called)

COMPARED TO RANDOM CALLING:
- Random calling would find only {int(total_calls * 0.117)} subscribers
- Our model found {found_yes} subscribers
- Improvement: {found_yes - int(total_calls * 0.117)} more subscribers found

CONCLUSION: The model is WORKING and should be DEPLOYED.
""")

--------------------------------------------------------------------------------------------------------------------------------------------
CONFUSION MATRIX
--------------------------------------------------------------------------------------------------------------------------------------------
Confusion Matrix:
                 Predicted NO    Predicted YES
Actual NO                4829            3156
Actual YES                 37            1021

HOW TO INTERPRET THE CONFUSION MATRIX

The confusion matrix shows four types of predictions:

+-----------------------------------------------------------+
|                       PREDICTED                           |
|                     NO           YES                      |
+-----------------------------------------------------------+
| ACTUAL    NO    [A]           [B]                         |
|           YES   [C]           [D]                         |
+-----------------------------------------------------------+

WHERE:
- [A] T

## Business Metrics Interpretation

### What These Numbers Mean for the Bank

| Metric | Value | Meaning |
|--------|-------|---------|
| Recall | 96.5% | For every 100 customers who would subscribe, the model identifies 97 of them |
| Precision | 24.4% | When the model recommends calling 100 customers, 24 actually subscribe |
| Accuracy | 64.7% | The model makes correct predictions for 65 out of every 100 customers |

---

### Understanding the Confusion Matrix Results

From our confusion matrix:

| Category | Count | Meaning |
|----------|-------|---------|
| True Negatives | 4,829 | Customers correctly identified as non-subscribers. The bank saves time by not calling them. |
| False Positives | 3,156 | Customers predicted to subscribe but did not. The bank wasted calls on these people. |
| False Negatives | 37 | Customers predicted not to subscribe but would have. The bank missed these opportunities. |
| True Positives | 1,021 | Customers correctly identified as subscribers. The bank successfully acquired these customers. |

---

### Impact on Bank Operations

**When the bank uses this model to guide calling decisions:**

| Metric | Value |
|--------|-------|
| Total calls made | 4,177 |
| Customers acquired | 1,021 |
| Wasted calls | 3,156 |
| Missed opportunities | 37 |

**Key observation:** Out of 1,058 total subscribers in the test data, the model identified 1,021. Only 37 subscribers were missed.

---

### Comparison with Random Calling

Random calling means selecting customers to call without any predictive model.

| Method | Subscribers Found (per 4,177 calls) |
|--------|--------------------------------------|
| Random Calling | 489 |
| Our Model | 1,021 |

**Result:** The model identifies 532 more subscribers than random selection.

---

### Model Assessment

The model meets all criteria for deployment:

| Criterion | Status | Evidence |
|-----------|--------|----------|
| High recall | PASSED | 96.5% of subscribers identified |
| Low missed opportunities | PASSED | Only 37 subscribers missed out of 1,058 |
| Better than random | PASSED | Identifies 532 more subscribers than random selection |
| Acceptable waste rate | PASSED | 3,156 wasted calls is within acceptable limits for bank call centers |

---

### Recommendation

**DEPLOY THIS MODEL**

The bank should integrate this model into its customer calling strategy. The model identifies 97% of potential subscribers while reducing wasted calls compared to random selection. The 3,156 wasted calls are acceptable given the 1,021 new customers acquired. The 37 missed opportunities represent a minimal loss rate of 3.5%.

### Saving evaluation report

In [12]:
### Saving evaluation report

print("-"*140)
print("SAVING EVALUATION REPORT")
print("-"*140)

os.makedirs("../outputs/reports", exist_ok=True)

# Save metrics to JSON
metrics = {
    "test_recall": float(recall),
    "test_precision": float(precision),
    "test_accuracy": float(accuracy),
    "test_f1": float(f1),
    "confusion_matrix": {
        "true_negatives": int(cm[0,0]),
        "false_positives": int(cm[0,1]),
        "false_negatives": int(cm[1,0]),
        "true_positives": int(cm[1,1])
    }
}

with open("../outputs/reports/model_performance_report.json", "w") as f:
    json.dump(metrics, f, indent=4)
print("✓ Performance report saved to: ../outputs/reports/model_performance_report.json")

# Save as text for easy reading
with open("../outputs/reports/model_performance_report.txt", "w") as f:
    f.write("-"*70 + "\n")
    f.write("MODEL PERFORMANCE REPORT\n")
    f.write("-"*70 + "\n\n")
    f.write(f"Test Recall:    {recall:.4f} ({recall*100:.1f}%)\n")
    f.write(f"Test Precision: {precision:.4f} ({precision*100:.1f}%)\n")
    f.write(f"Test Accuracy:  {accuracy:.4f} ({accuracy*100:.1f}%)\n")
    f.write(f"Test F1-Score:  {f1:.4f}\n\n")
    f.write("CONFUSION MATRIX:\n")
    f.write(f"  True Negatives:  {cm[0,0]}\n")
    f.write(f"  False Positives: {cm[0,1]}\n")
    f.write(f"  False Negatives: {cm[1,0]}\n")
    f.write(f"  True Positives:  {cm[1,1]}\n")

print("✓ Text report saved to: ../outputs/reports/model_performance_report.txt")

print("\n" + "-"*140)
print("NOTEBOOK 07 - PART 1 COMPLETED")
print("-"*140)

--------------------------------------------------------------------------------------------------------------------------------------------
SAVING EVALUATION REPORT
--------------------------------------------------------------------------------------------------------------------------------------------
✓ Performance report saved to: ../outputs/reports/model_performance_report.json
✓ Text report saved to: ../outputs/reports/model_performance_report.txt

--------------------------------------------------------------------------------------------------------------------------------------------
NOTEBOOK 07 - PART 1 COMPLETED
--------------------------------------------------------------------------------------------------------------------------------------------


### Deployment readiness check

In [13]:
### Deployment readiness check

print("-"*140)
print("DEPLOYMENT READINESS CHECK")
print("-"*140)

checks = []

# Check model exists
if os.path.exists("../outputs/models/best_model.pkl"):
    checks.append("PASS: Model file exists")
else:
    checks.append("FAIL: Model file missing")

# Check preprocessor exists
if os.path.exists("../outputs/models/preprocessor.pkl"):
    checks.append("PASS: Preprocessor file exists")
else:
    checks.append("WARNING: Preprocessor file missing")

# Check metrics report exists
if os.path.exists("../outputs/reports/model_performance_report.json"):
    checks.append("PASS: Performance report exists")
else:
    checks.append("FAIL: Performance report missing")

# Check recall is acceptable (>= 90%)
if recall >= 0.90:
    checks.append(f"PASS: Recall is {recall*100:.1f}% (target is 90%)")
else:
    checks.append(f"FAIL: Recall is {recall*100:.1f}% (target is 90%)")

# Check precision is acceptable (>= 20%)
if precision >= 0.20:
    checks.append(f"PASS: Precision is {precision*100:.1f}% (target is 20%)")
else:
    checks.append(f"WARNING: Precision is {precision*100:.1f}% (target is 20%)")

print("\n".join(checks))

print("\n" + "="*70)
print("DEPLOYMENT STATUS")
print("="*70)

# Count passes and failures
pass_count = sum(1 for c in checks if c.startswith("PASS"))
fail_count = sum(1 for c in checks if c.startswith("FAIL"))
warning_count = sum(1 for c in checks if c.startswith("WARNING"))

print(f"Passes: {pass_count}")
print(f"Failures: {fail_count}")
print(f"Warnings: {warning_count}")

if fail_count == 0:
    print("\nSTATUS: READY FOR DEPLOYMENT")
    print("All critical checks passed. The model can be deployed to production.")
else:
    print("\nSTATUS: NOT READY FOR DEPLOYMENT")
    print("Please address the failures before proceeding.")

print("\n" + "-"*140)
print("NOTEBOOK 07 COMPLETED SUCCESSFULLY")
print("-"*140)
print("\nModel evaluation complete. Ready for deployment documentation.")

--------------------------------------------------------------------------------------------------------------------------------------------
DEPLOYMENT READINESS CHECK
--------------------------------------------------------------------------------------------------------------------------------------------
PASS: Model file exists
PASS: Preprocessor file exists
PASS: Performance report exists
PASS: Recall is 96.5% (target is 90%)
PASS: Precision is 24.4% (target is 20%)

DEPLOYMENT STATUS
Passes: 5
Failures: 0
Warnings: 0

STATUS: READY FOR DEPLOYMENT
All critical checks passed. The model can be deployed to production.

--------------------------------------------------------------------------------------------------------------------------------------------
NOTEBOOK 07 COMPLETED SUCCESSFULLY
--------------------------------------------------------------------------------------------------------------------------------------------

Model evaluation complete. Ready for deployment docume